# 🧪 ALCheMy — Probador Interactivo del Webhook de n8n

Este notebook tiene dos objetivos:

1. **Probar todas las casuísticas del pipeline de ingesta**: un caso "happy path" por cada una de las 9 técnicas deterministas/microscopía + las 2 técnicas vía IA (PDF), y todos los fallbacks (email no autorizado, tamaño excedido, duplicado, sample_code ausente, técnica no reconocida, extensión inválida, rechazo humano L2, timeout de aprobación, fallo del agente de IA).
2. **Definir el esqueleto de un cliente Python** (`ALCheMyIngestionClient`) para el webhook de ingesta — la base de lo que sería una API pública real: hoy habla HTTP directamente contra n8n, pero el contrato (`submit(file_path, technique, sample_code, ...)`) es el mismo que expondría un futuro wrapper en FastAPI u otro backend.

## ⚠️ Antes de ejecutar

- `docker compose up -d` debe estar corriendo (n8n, runners, PostgreSQL).
- El workflow **ALCheMy** debe estar **activo** en n8n, y el nodo `Webhook` **habilitado** — si seguiste esta sesión, ya quedó así (antes estaba desactivado por defecto). Si en algún momento lo desactivas desde el editor, este notebook dejará de recibir respuesta en `mode="prod"`.
- Recarga (F5) tu pestaña de n8n antes de ejecutar esto, para no arrastrar una versión desincronizada del workflow.
- Instala dependencias si te faltan: `pip install requests psycopg2-binary python-dotenv`.


In [ ]:
import os
import json
import time
import tempfile
import requests
import psycopg2

from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_LAKE = REPO_ROOT / "data" / "raw_datalake_historico"

# Conexion a PostgreSQL - mismos defaults que src/ingest_datalake_to_postgres.py
DB_HOST = os.environ.get("POSTGRES_HOST", "localhost")
DB_PORT = int(os.environ.get("POSTGRES_PORT", 5432))
DB_NAME = os.environ.get("POSTGRES_DB", "alchemy_db")
DB_USER = os.environ.get("POSTGRES_USER", "alchemy_user")
DB_PASS = os.environ["POSTGRES_PASSWORD"]  # sin valor por defecto: nunca hardcodear una contrasena real aqui

def get_db_connection():
    return psycopg2.connect(host=DB_HOST, port=DB_PORT, dbname=DB_NAME, user=DB_USER, password=DB_PASS)

print("Repo root:", REPO_ROOT)
print("Data lake:", DATA_LAKE, "existe:", DATA_LAKE.exists())


## 🧩 Esqueleto del cliente / API de ingesta

`ALCheMyIngestionClient` envuelve el POST multipart al webhook de n8n. Es intencionadamente
delgado: un solo metodo publico (`submit`), sin logica de negocio propia (esa vive en el
workflow). Si en el futuro se monta una API propia delante de n8n (p.ej. FastAPI para dar
autenticacion real, rate limiting, o validacion previa), el contrato de `submit(...)` es el
que se conservaria - solo cambiaria el transporte interno.


In [ ]:
class ALCheMyIngestionClient:
    """Cliente minimo para el webhook de ingesta de caracterizaciones de ALCheMy.

    mode="prod"  -> usa la URL de produccion (/webhook/...), requiere el workflow ACTIVO.
    mode="test"  -> usa la URL de test (/webhook-test/...), requiere pulsar "Listen for
                    test event" en el editor de n8n antes de cada llamada (no apto para
                    baterias de pruebas automatizadas sin supervision).
    """

    def __init__(self, base_url: str = "http://localhost:5678", mode: str = "prod"):
        self.base_url = base_url.rstrip("/")
        self.mode = mode

    @property
    def endpoint(self) -> str:
        segment = "webhook" if self.mode == "prod" else "webhook-test"
        return f"{self.base_url}/{segment}/ingest-characterization"

    def submit(
        self,
        file_path,
        technique: str,
        sample_code: str = None,
        researcher_email: str = "rubbenserrano@gmail.com",
        researcher_name: str = "Ruben Serrano",
        operator_name: str = None,
        measurement_tag: str = None,
        instrument_metadata_file=None,
        timeout: int = 60,
    ) -> requests.Response:
        data = {
            "technique": technique,
            "researcher_email": researcher_email,
            "researcher_name": researcher_name,
        }
        if sample_code is not None:
            data["sample_code"] = sample_code
        if operator_name:
            data["operator_name"] = operator_name
        if measurement_tag:
            data["measurement_tag"] = measurement_tag

        file_path = str(file_path)
        f_main = open(file_path, "rb")
        # n8n's multipart parser only routes a part to $binary when it carries a
        # Content-Type header - requests only adds one automatically when Python's
        # mimetypes module recognizes the extension, which it does NOT for .xrdml/.brml/
        # .asc/etc. Without it, n8n silently decodes the file as a UTF-8 text field
        # into body.data_file instead of $binary, so Check File Size sees no size at
        # all and rejects it as FILE_SIZE_EXCEEDED. Force a generic binary Content-Type
        # explicitly so this never depends on the extension being a "known" one.
        files = {"data_file": (os.path.basename(file_path), f_main, "application/octet-stream")}

        f_meta = None
        if instrument_metadata_file:
            f_meta = open(str(instrument_metadata_file), "rb")
            files["instrument_metadata_file"] = (os.path.basename(str(instrument_metadata_file)), f_meta, "application/octet-stream")

        try:
            return requests.post(self.endpoint, data=data, files=files, timeout=timeout)
        finally:
            f_main.close()
            if f_meta:
                f_meta.close()


client = ALCheMyIngestionClient(mode="prod")
print("Endpoint activo:", client.endpoint)


In [ ]:
def run_case(title, response, expect_status=None, expect_cause_contains=None):
    """Imprime el resultado de una llamada y valida las expectativas basicas."""
    print(f"\n{'='*78}\n{title}\n{'='*78}")
    print(f"HTTP {response.status_code}")
    body = {}
    try:
        body = response.json()
        print(json.dumps(body, indent=2, ensure_ascii=False))
    except Exception:
        print(response.text[:500])

    ok = True
    if expect_status is not None and response.status_code != expect_status:
        ok = False
        print(f"[FALLO] Se esperaba HTTP {expect_status}, llego {response.status_code}")
    if expect_cause_contains is not None:
        cause = body.get("cause", "") if isinstance(body, dict) else ""
        if expect_cause_contains not in cause:
            ok = False
            print(f"[FALLO] La causa no contiene '{expect_cause_contains}'. Causa real: {cause!r}")
    print("[OK]" if ok else "[REVISAR]")
    return body


---
## ✅ Parte 1 — Happy path por tecnica

Un caso por cada una de las 11 combinaciones tecnica+ruta soportadas: 6 parsers deterministas
"clasicos", 3 de microscopia (SEM/FESEM/TEM, con el caso FESEM demostrando el nuevo campo
opcional de metadatos del instrumento), y 2 via agente de IA sobre PDF (ICP-OES, EA).

Cada celda espera HTTP 200 y `status: success`. Si un archivo real cambia de nombre en el
data lake, ajusta la ruta.


### PXRD (.xrdml)

In [ ]:
resp = client.submit(
    file_path=DATA_LAKE / "PXRD" / "24DRX0180-RS1883_CoPy.xrdml",
    technique="PXRD",
    sample_code="RS1883",
)
_ = run_case("Happy path - PXRD", resp, expect_status=200)


### VT-XRD (.brml)

In [ ]:
resp = client.submit(
    file_path=DATA_LAKE / "VT-XRD" / "24DRX0766-RSMix15_CoPyF2_1 Air100mlmin.brml",
    technique="VT-XRD",
    sample_code="RSMix15",
)
_ = run_case("Happy path - VT-XRD", resp, expect_status=200)


### TGA (.txt)

In [ ]:
resp = client.submit(
    file_path=DATA_LAKE / "TGA" / "IEF-22(Fe)_Ar_1000ºC_RS1104.txt",
    technique="TGA",
    sample_code="RS1104",
)
_ = run_case("Happy path - TGA", resp, expect_status=200)


### FTIR (.csv)

In [ ]:
resp = client.submit(
    file_path=DATA_LAKE / "FTIR" / "CoPyF1.CSV",
    technique="FTIR",
    sample_code="RS-FTIR-DEMO",
)
_ = run_case("Happy path - FTIR", resp, expect_status=200)


### Adsorcion / BET (.xls)

In [ ]:
resp = client.submit(
    file_path=DATA_LAKE / "ADSORPTION" / "20220425-22046-RS522-Mg-Tria7-st2.XLS",
    technique="Adsorption",
    sample_code="RS522",
)
_ = run_case("Happy path - Adsorcion", resp, expect_status=200)


### UV-Vis (.asc)

In [ ]:
resp = client.submit(
    file_path=DATA_LAKE / "UV_Vis" / "IEF-8(CB-12A).Sample.asc",
    technique="UV-Vis",
    sample_code="IEF-8",
)
_ = run_case("Happy path - UV-Vis", resp, expect_status=200)


### SEM (.jpg + .txt Hitachi)

Sube la micrografia **y** su `.txt` companero. Entre los tres casos de microscopia quedan
cubiertos los tres formatos reales de metadatos que existen en el data lake:

| Caso  | Formato del sidecar | De donde sale el aumento |
|-------|---------------------|--------------------------|
| SEM   | Hitachi INI (`[SemImageFile]`, `Clave=Valor`) | del sidecar |
| FESEM | JEOL (`$CM_MAG`, `$$SM_WD`...) | del sidecar |
| TEM   | no hay sidecar | del nombre del archivo (`_MAG_X3000_`) |


In [ ]:
resp = client.submit(
    file_path=DATA_LAKE / "SEM" / "IEF-18(Co)0004(x1.0k).jpg",
    technique="SEM",
    sample_code="IEF-18",
    instrument_metadata_file=DATA_LAKE / "SEM" / "IEF-18(Co)0004(x1.0k).txt",
)
_ = run_case("Happy path - SEM (con .txt Hitachi)", resp, expect_status=200)


### FESEM (.bmp, CON metadatos adjuntos)

Este es el caso que demuestra el campo nuevo: subimos la imagen JEOL (`.bmp`) **y** su `.txt`
compañero a la vez via `instrument_metadata_file`. El parser determinista debe extraer
instrumento real (`JEOL 7900F`), aumento, voltaje, y las anotaciones de cursor del operador.


In [ ]:
resp = client.submit(
    file_path=DATA_LAKE / "FESEM" / "IEF-31 + PCL" / "IEF-31+PCL(1)" / "IEF-31+PCL(1)_1.bmp",
    technique="FESEM",
    sample_code="IEF-31",
    instrument_metadata_file=DATA_LAKE / "FESEM" / "IEF-31 + PCL" / "IEF-31+PCL(1)" / "IEF-31+PCL(1)_1.txt",
)
_ = run_case("Happy path - FESEM (con .txt)", resp, expect_status=200)


### TEM (.jpg, sin sidecar - aumento en el nombre de archivo)

TEM nunca trae `.txt`; el aumento se lee del propio nombre de archivo
(`..._MAG_X3000_001.jpg`), no de un sidecar.


In [ ]:
resp = client.submit(
    file_path=DATA_LAKE / "TEM" / "IEF-31-PCL-Lavado" / "IEF-31-PCL-Lavado_MAG_X3000_001.jpg",
    technique="TEM",
    sample_code="IEF-31",
)
_ = run_case("Happy path - TEM", resp, expect_status=200)


### ICP-OES (.pdf, via Agente de IA)

Esta rama pasa por el agente de IA (Gemini) y el evaluador de confianza. Si la confianza
declarada es >= 85%% inserta directamente; si es menor, dispara el email de aprobacion L2
(ver Parte 2, casos de rechazo humano / timeout).


In [ ]:
resp = client.submit(
    file_path=DATA_LAKE / "ICP" / "24AQ0245ICP%20Report_Fe,%20Ni,%20P_MOFs_Rubén%20Serrano.pdf",
    technique="ICP-OES",
    sample_code="IEF-18",
)
_ = run_case("Happy path - ICP-OES (via IA)", resp, expect_status=200)


### EA / CHNS (.pdf, via Agente de IA)

In [ ]:
resp = client.submit(
    file_path=DATA_LAKE / "EA" / "CHNS-23047-Rubén.pdf",
    technique="EA",
    sample_code="IEF-18",
)
_ = run_case("Happy path - EA (via IA)", resp, expect_status=200)


---
## 🧯 Parte 2 — Casuisticas de fallback

Cada fallback escribe en `audit_log` con `verdict/cause/result` tipados
(`workflows/error_payloads_matrix.csv`). Los primeros 7 casos son totalmente automatizables
desde este notebook; los 3 ultimos (rechazo humano, timeout de aprobacion, fallo del agente
de IA) dependen de una interaccion humana o de un temporizador real y se documentan en vez
de ejecutarse ciegamente - forzarlos "de mentira" en un notebook daria una falsa sensacion
de cobertura.


### 1. Email no autorizado (whitelist)

`Auto-Enroll Whitelist` auto-registra cualquier email nuevo como activo (self-enrollment de
demo) - pero **no** reactiva un email que un admin haya desactivado explicitamente
(`ON CONFLICT ... DO UPDATE SET last_seen = ...` no toca `active`). Por eso, para probar este
fallback primero hay que bloquear un email a mano.


In [ ]:
BLOCKED_EMAIL = "bloqueado.test@example.com"

def block_email(email):
    conn = get_db_connection()
    try:
        with conn.cursor() as cur:
            cur.execute(
                """
                INSERT INTO authorized_researchers (email, researcher_name, role, active)
                VALUES (%s, 'Blocked Test User', 'blocked_test', false)
                ON CONFLICT (email) DO UPDATE SET active = false;
                """,
                (email,),
            )
        conn.commit()
    finally:
        conn.close()

block_email(BLOCKED_EMAIL)
print(f"{BLOCKED_EMAIL} marcado como active=false en authorized_researchers")


In [ ]:
resp = client.submit(
    file_path=DATA_LAKE / "PXRD" / "24DRX0180-RS1883_CoPy.xrdml",
    technique="PXRD",
    sample_code="RS1883",
    researcher_email=BLOCKED_EMAIL,
)
_ = run_case("Fallback - Email no autorizado", resp, expect_status=400, expect_cause_contains="UNAUTHORIZED_RESEARCHER_EMAIL")


### 2. Tamaño de archivo excedido (> 10 MB)

Generamos un archivo sintetico de relleno en vez de depender de que exista uno real de mas
de 10MB en el data lake.


In [ ]:
oversized_path = Path(tempfile.gettempdir()) / "alchemy_test_oversized.xrdml"
with open(oversized_path, "wb") as f:
    f.write(b"0" * (11 * 1024 * 1024))  # 11 MB > limite de 10 MB (10485760 bytes)

resp = client.submit(
    file_path=oversized_path,
    technique="PXRD",
    sample_code="RS-OVERSIZED-TEST",
)
_ = run_case("Fallback - Tamaño excedido", resp, expect_status=400, expect_cause_contains="FILE_SIZE_EXCEEDED")

oversized_path.unlink(missing_ok=True)


### 3. Medida duplicada

Reenvia el mismo archivo PXRD del happy path (**ejecuta primero la celda de PXRD de la Parte
1**, si no este caso no tendra nada de lo que ser duplicado).


In [ ]:
resp = client.submit(
    file_path=DATA_LAKE / "PXRD" / "24DRX0180-RS1883_CoPy.xrdml",
    technique="PXRD",
    sample_code="RS1883",
)
_ = run_case("Fallback - Duplicado", resp, expect_status=400, expect_cause_contains="DUPLICATE_MEASUREMENT_SKIPPED")


### 4. sample_code ausente

No se envia `sample_code`, y el nombre de archivo no contiene ningun patron reconocible
(`RS\d+`, `IEF-\d+`, `RSMix\d+`, `LP\d+`).


In [ ]:
resp = client.submit(
    file_path=DATA_LAKE / "FTIR" / "CoPyF2.CSV",
    technique="FTIR",
    sample_code="",
)
_ = run_case("Fallback - sample_code ausente", resp, expect_status=400, expect_cause_contains="MISSING_SAMPLE_CODE")


### 5. Tecnica no reconocida

In [ ]:
resp = client.submit(
    file_path=DATA_LAKE / "PXRD" / "24DRX0180-RS1883_CoPy.xrdml",
    technique="MASS_SPEC_XYZ",
    sample_code="RS-TECH-TEST",
)
_ = run_case("Fallback - Tecnica no reconocida", resp, expect_status=400, expect_cause_contains="UNRECOGNIZED_TECHNIQUE")


### 6. Extension invalida para una tecnica determinista

Se declara `technique="PXRD"` pero se sube un `.csv` de FTIR - el router determinista
(tecnica+extension) no tiene ninguna regla que lo acepte.


In [ ]:
resp = client.submit(
    file_path=DATA_LAKE / "FTIR" / "CoPyF1.CSV",
    technique="PXRD",
    sample_code="RS-EXT-TEST-1",
)
_ = run_case("Fallback - Extension invalida (determinista)", resp, expect_status=400, expect_cause_contains="INVALID_FILE_EXTENSION")


### 7. Extension invalida para una tecnica via IA (PDF)

Se declara `technique="ICP-OES"` pero se sube un `.xrdml` en vez de un `.pdf`.


In [ ]:
resp = client.submit(
    file_path=DATA_LAKE / "PXRD" / "24DRX0180-RS1883_CoPy.xrdml",
    technique="ICP-OES",
    sample_code="RS-EXT-TEST-2",
)
_ = run_case("Fallback - Extension invalida (PDF)", resp, expect_status=400, expect_cause_contains="INVALID_FILE_EXTENSION")


### 8. Rechazo humano en la aprobacion L2 (manual)

No se puede automatizar desde aqui: hay que dispararlo y luego pulsar **Decline** en el
correo de aprobacion que llega a `rubbenserrano@gmail.com` (o a quien configures como
`researcher_email`).

1. Ejecuta la celda de abajo para disparar una extraccion via IA.
2. Si la confianza del agente sale por debajo del 85%%, te llegara un correo de aprobacion
   con botones **Approve** / **Disapprove**.
3. Pulsa **Disapprove**. El flujo deberia registrar en `audit_log` la causa
   `REJECTED_BY_RESEARCHER` y no insertar la medida.
4. Si la confianza sale por encima del 85%%, no hay aprobacion que rechazar - vuelve a
   intentarlo con otro PDF, o reduce el `CONFIDENCE_THRESHOLD` temporalmente en el nodo
   `Confidence Evaluator` solo para esta prueba.


In [ ]:
resp = client.submit(
    file_path=DATA_LAKE / "EA" / "CHNS-21008-Catalina.pdf",
    technique="EA",
    sample_code="IEF-18",
)
_ = run_case("Disparo manual - posible aprobacion L2 (revisa tu email)", resp)


### 9. Timeout de aprobacion (manual, dependiente del tiempo)

Mismo disparador que el caso anterior, pero esta vez **no respondas al correo**. Pasados
5 minutos (`resumeAmount: 5` en el nodo `Send message and wait for response`), el flujo
deberia seguir por la rama de error automaticamente, con causa `APPROVAL_TIMEOUT_OR_ERROR`.
No lo esperamos de forma bloqueante en una celda - revisa `audit_log` pasados los 5 minutos:

```sql
SELECT * FROM audit_log ORDER BY created_at DESC LIMIT 5;
```


In [ ]:
resp = client.submit(
    file_path=DATA_LAKE / "ICP" / "24AQ0329ICP%20Report_Ca,%20Co,%20P_MOFs_Rubén%20Serrano.pdf",
    technique="ICP-OES",
    sample_code="IEF-18",
)
_ = run_case("Disparo manual - posible timeout de aprobacion (no respondas al email)", resp)


### 10. Fallo del agente de IA (manual / con manipulacion de entorno)

Para forzar este fallback de forma fiable hay que romper deliberadamente la llamada al
modelo, por ejemplo:

1. En `.env`, pon `GEMINI_API_KEY` a un valor invalido (guarda el valor real antes).
2. `docker compose restart n8n`.
3. Ejecuta la celda de abajo (cualquier PDF o imagen sirve).
4. Deberia llegar HTTP 400 con causa `AI_AGENT_EXTRACTION_FAILED`, registrado en `audit_log`.
5. Restaura la `GEMINI_API_KEY` real y reinicia n8n otra vez.

No se ejecuta automaticamente aqui porque manipular `.env` y reiniciar contenedores desde un
notebook es mas invasivo de lo que conviene automatizar sin supervision directa.


In [ ]:
# resp = client.submit(
#     file_path=DATA_LAKE / "ICP" / "24AQ0446ICP%20Report_Barrido_Agua%20residual_Rubén%20Serrano.pdf",
#     technique="ICP-OES",
#     sample_code="IEF-18",
# )
# _ = run_case("Fallback - Fallo del agente de IA", resp, expect_status=400, expect_cause_contains="AI_AGENT_EXTRACTION_FAILED")


---
## 🔎 Verificacion final en `audit_log`

Consulta rapida para revisar de un vistazo todo lo que este notebook acaba de generar.


In [ ]:
conn = get_db_connection()
try:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT created_at, agent, action, verdict, cause, result
            FROM audit_log
            ORDER BY created_at DESC
            LIMIT 25;
        """)
        rows = cur.fetchall()
        cols = [d.name for d in cur.description]
finally:
    conn.close()

import pandas as pd
pd.DataFrame(rows, columns=cols)


---
## 🤖 Verificacion automatizada (todo de una vez)

Las celdas de arriba son para inspeccionar **un** caso concreto con calma. Para comprobar
que **todo** el pipeline sigue sano hay un script que dispara los 9 happy paths y los 7
fallbacks automatizables de golpe, y que ademas verifica que la fila aterrizo en
`characterization_measurements` con los metadatos que el parser debia extraer (no solo que
el HTTP devolvio 200):

```bash
python src/verify_webhook_pipeline.py                # todo
python src/verify_webhook_pipeline.py --happy        # solo happy paths
python src/verify_webhook_pipeline.py --fallbacks    # solo fallbacks
python src/verify_webhook_pipeline.py --only PXRD    # un caso suelto
```

Existe porque el pipeline tiene muchas superficies de fallo independientes (el sandbox de
Python de n8n, el manejo de binarios multipart, el router tecnica+extension, las FK de
Postgres) y probarlas de una en una a mano es justo como se colaron las regresiones: un
arreglo para una tecnica rompia o tapaba otra sin que se notase.

In [ ]:
import subprocess, sys

# Ejecuta la bateria completa. Devuelve codigo 0 solo si pasan los 16 casos.
proc = subprocess.run(
    [sys.executable, str(REPO_ROOT / "src" / "verify_webhook_pipeline.py")],
    capture_output=True, text=True, cwd=str(REPO_ROOT),
)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr[-2000:])


---
## ⚠️ Limites del sandbox de Python de n8n (aprendidos a base de golpes)

Los nodos **Code** en Python no corren en un Python normal, sino en un sandbox que bloquea
cosas sin avisar de forma clara. Todo esto esta verificado empiricamente contra el runner
real, no supuesto:

| Que | Sintoma | Como convivir con ello |
|-----|---------|------------------------|
| `open` a secas esta prohibido | `name 'open' is not defined` | `import io; open = io.open` al principio del nodo |
| `open(...)` en **modo texto** (`'r'`) esta roto | `'NoneType' object is not callable` | leer en binario (`'rb'`) y hacer `.decode(...)` a mano |
| `getattr` y `type` estan prohibidos | `name 'getattr' is not defined` | evitarlos; usar `try/except` y `str(e)` |
| Los binarios llegan como marcador, no como datos | `base64.b64decode` produce basura | convertir antes con un nodo `Move Binary Data` (`keepAsBase64`) |

Y un detalle del **nodo Webhook**, no del sandbox: numera *todos* los archivos subidos bajo
el mismo prefijo (`data_file0`, `data_file1`...), sin respetar el nombre del campo del
formulario. Por eso el `.txt` de metadatos llega como `data_file1` y hay que renombrarlo en
`Normalize Webhook Body`.

Nota de mantenimiento: los parsers viven **dos veces** (en `src/parse_*.py` y pegados dentro
de los nodos Code). Mantenerlos sincronizados a mano es como la copia de microscopia en n8n
se quedo meses atras sin soporte JEOL. Para sincronizar:

```bash
python src/sync_parsers_to_n8n.py --dry-run   # ver que ha divergido
python src/sync_parsers_to_n8n.py             # empujar src/ -> n8n
```